In [ ]:
!pip install gdown
!gdown 1JbUTyZZNJ1pRl8cHqRRodJPwLQkMGu1b

Downloading...
From (original): https://drive.google.com/uc?id=1JbUTyZZNJ1pRl8cHqRRodJPwLQkMGu1b
From (redirected): https://drive.google.com/uc?id=1JbUTyZZNJ1pRl8cHqRRodJPwLQkMGu1b&confirm=t&uuid=7d5111c2-947b-4ae5-979e-3c3c5a52fbdc
To: /content/Offroad_Segmentation_Training_Dataset.zip
 92% 2.56G/2.79G [00:13<00:01, 206MB/s]

In [ ]:
!pip install gdown

In [ ]:
import gdown

files = [
    "1JbUTyZZNJ1pRl8cHqRRodJPwLQkMGu1b",
    "1ziy5Ipcxzbxkr5mHLcYiCy3RwBt_9dUw",
    "1Vn7bqCt2UvwK2FU3BOdsNXK5IHPDDa1J"
]

for f in files:
    gdown.download(f"https://drive.google.com/uc?id={f}", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1JbUTyZZNJ1pRl8cHqRRodJPwLQkMGu1b
From (redirected): https://drive.google.com/uc?id=1JbUTyZZNJ1pRl8cHqRRodJPwLQkMGu1b&confirm=t&uuid=80b72727-6d15-45a3-a293-b54c2d191589
To: /content/Offroad_Segmentation_Training_Dataset.zip
100%|██████████| 2.79G/2.79G [00:32<00:00, 84.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ziy5Ipcxzbxkr5mHLcYiCy3RwBt_9dUw
From (redirected): https://drive.google.com/uc?id=1ziy5Ipcxzbxkr5mHLcYiCy3RwBt_9dUw&confirm=t&uuid=4f6f48db-73d2-42b9-a194-56580ef7c187
To: /content/Offroad_Segmentation_testImages.zip
100%|██████████| 1.16G/1.16G [00:14<00:00, 78.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Vn7bqCt2UvwK2FU3BOdsNXK5IHPDDa1J
To: /content/Offroad_Segmentation_Scripts.zip
100%|██████████| 12.8k/12.8k [00:00<00:00, 32.9MB/s]


In [ ]:
import zipfile
import os

for file in os.listdir():
    if file.endswith(".zip"):
        print(f"Unzipping {file}...")
        with zipfile.ZipFile(file, 'r') as zip_ref:
            zip_ref.extractall(file.replace(".zip",""))

print("All zip files extracted.")

Unzipping Offroad_Segmentation_Training_Dataset.zip...
Unzipping Offroad_Segmentation_testImages.zip...
Unzipping Offroad_Segmentation_Scripts.zip...
All zip files extracted.


In [ ]:
import os

for file in os.listdir():
    if file.endswith(".zip"):
        os.remove(file)
        print(f"Deleted {file}")

print("All zip files deleted.")

Deleted Offroad_Segmentation_Training_Dataset.zip
Deleted Offroad_Segmentation_testImages.zip
Deleted Offroad_Segmentation_Scripts.zip
All zip files deleted.


In [ ]:
!mv Offroad_Segmentation_Training_Dataset Offroad_Segmentation_Scripts/
!mv Offroad_Segmentation_testImages Offroad_Segmentation_Scripts/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp -r /content/Offroad_Segmentation_Scripts /content/drive/MyDrive/HAWKATHON

ENV_SETUP			       test_segmentation.py
Offroad_Segmentation_testImages        train_segmentation.py
Offroad_Segmentation_Training_Dataset  visualize.py


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!ls


drive  Offroad_Segmentation_Scripts  sample_data


In [ ]:
%cd Offroad_Segmentation_Scripts
!ls

/content/Offroad_Segmentation_Scripts
ENV_SETUP			       test_segmentation.py
Offroad_Segmentation_testImages        train_segmentation.py
Offroad_Segmentation_Training_Dataset  visualize.py


In [ ]:
!pip install torch torchvision
!pip install opencv-contrib-python
!pip install tqdm matplotlib pillow numpy

In [ ]:
import torch
torch.hub.set_dir('/content/torchhub')

In [ ]:
%cd /content

!mv Offroad_Segmentation_Scripts/Offroad_Segmentation_Training_Dataset .
!mv Offroad_Segmentation_Scripts/Offroad_Segmentation_testImages .

/content


In [ ]:
!ls

drive				 Offroad_Segmentation_Training_Dataset
Offroad_Segmentation_Scripts	 sample_data
Offroad_Segmentation_testImages


In [ ]:
!ls /content/Offroad_Segmentation_Training_Dataset

Offroad_Segmentation_Training_Dataset


In [ ]:
!mv /content/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/* /content/Offroad_Segmentation_Training_Dataset/

In [ ]:
!rm -r /content/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset

In [ ]:
!ls /content/Offroad_Segmentation_Training_Dataset

train  val


In [ ]:
!ls /content/Offroad_Segmentation_Training_Dataset/train

Color_Images  Segmentation


In [ ]:
!ls

ENV_SETUP	      train_segmentation.py  visualize.py
test_segmentation.py  train_stats


In [ ]:
%%writefile train_segmentation.py
"""
Segmentation Training Script
Converted from train_mask.ipynb
Trains a segmentation head on top of DINOv2 backbone
"""

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
import cv2
import os
import torchvision
from tqdm import tqdm

# Set matplotlib to non-interactive backend
plt.switch_backend('Agg')


# ============================================================================
# Utility Functions
# ============================================================================

def save_image(img, filename):
    """Save an image tensor to file after denormalizing."""
    img = np.array(img)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = np.moveaxis(img, 0, -1)
    img = (img * std + mean) * 255
    cv2.imwrite(filename, img[:, :, ::-1])


# ============================================================================
# Mask Conversion
# ============================================================================

# Mapping from raw pixel values to new class IDs
value_map = {
    0: 0,        # background
    100: 1,      # Trees
    200: 2,      # Lush Bushes
    300: 3,      # Dry Grass
    500: 4,      # Dry Bushes
    550: 5,      # Ground Clutter
    700: 6,      # Logs
    800: 7,      # Rocks
    7100: 8,     # Landscape
    10000: 9     # Sky
}
n_classes = len(value_map)


def convert_mask(mask):
    """Convert raw mask values to class IDs."""
    arr = np.array(mask)
    new_arr = np.zeros_like(arr, dtype=np.uint8)
    for raw_value, new_value in value_map.items():
        new_arr[arr == raw_value] = new_value
    return Image.fromarray(new_arr)


# ============================================================================
# Dataset
# ============================================================================

class MaskDataset(Dataset):
    def __init__(self, data_dir, transform=None, mask_transform=None):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.masks_dir = os.path.join(data_dir, 'Segmentation')
        self.transform = transform
        self.mask_transform = mask_transform
        self.data_ids = os.listdir(self.image_dir)

    def __len__(self):
        return len(self.data_ids)

    def __getitem__(self, idx):
        data_id = self.data_ids[idx]
        img_path = os.path.join(self.image_dir, data_id)
        # Both color images and masks are .png files with same name
        mask_path = os.path.join(self.masks_dir, data_id)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        mask = convert_mask(mask)

        if self.transform:
            image = self.transform(image)
            mask = self.mask_transform(mask) * 255

        return image, mask


# ============================================================================
# Model: Segmentation Head (ConvNeXt-style)
# ============================================================================

class SegmentationHeadConvNeXt(nn.Module):
    def __init__(self, in_channels, out_channels, tokenW, tokenH):
        super().__init__()
        self.H, self.W = tokenH, tokenW

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=7, padding=3),
            nn.GELU()
        )

        self.block = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=7, padding=3, groups=128),
            nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=1),
            nn.GELU(),
        )

        self.classifier = nn.Conv2d(128, out_channels, 1)

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.H, self.W, C).permute(0, 3, 1, 2)
        x = self.stem(x)
        x = self.block(x)
        return self.classifier(x)


# ============================================================================
# Metrics
# ============================================================================

def compute_iou(pred, target, num_classes=10, ignore_index=255):
    """Compute IoU for each class and return mean IoU."""
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)

    iou_per_class = []
    for class_id in range(num_classes):
        if class_id == ignore_index:
            continue

        pred_inds = pred == class_id
        target_inds = target == class_id

        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()

        if union == 0:
            iou_per_class.append(float('nan'))
        else:
            iou_per_class.append((intersection / union).cpu().numpy())

    return np.nanmean(iou_per_class)


def compute_dice(pred, target, num_classes=10, smooth=1e-6):
    """Compute Dice coefficient (F1 Score) per class and return mean Dice Score."""
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)

    dice_per_class = []
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id

        intersection = (pred_inds & target_inds).sum().float()
        dice_score = (2. * intersection + smooth) / (pred_inds.sum().float() + target_inds.sum().float() + smooth)

        dice_per_class.append(dice_score.cpu().numpy())

    return np.mean(dice_per_class)


def compute_pixel_accuracy(pred, target):
    """Compute pixel accuracy."""
    pred_classes = torch.argmax(pred, dim=1)
    return (pred_classes == target).float().mean().cpu().numpy()


def evaluate_metrics(model, backbone, data_loader, device, num_classes=10, show_progress=True):
    """Evaluate all metrics on a dataset."""
    iou_scores = []
    dice_scores = []
    pixel_accuracies = []

    model.eval()
    loader = tqdm(data_loader, desc="Evaluating", leave=False, unit="batch") if show_progress else data_loader
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            output = backbone.forward_features(imgs)["x_norm_patchtokens"]
            logits = model(output.to(device))
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode="bilinear", align_corners=False)

            labels = labels.squeeze(dim=1).long()

            iou = compute_iou(outputs, labels, num_classes=num_classes)
            dice = compute_dice(outputs, labels, num_classes=num_classes)
            pixel_acc = compute_pixel_accuracy(outputs, labels)

            iou_scores.append(iou)
            dice_scores.append(dice)
            pixel_accuracies.append(pixel_acc)

    model.train()
    return np.mean(iou_scores), np.mean(dice_scores), np.mean(pixel_accuracies)


# ============================================================================
# Plotting Functions
# ============================================================================

def save_training_plots(history, output_dir):
    """Save all training metric plots to files."""
    os.makedirs(output_dir, exist_ok=True)

    # Plot 1: Loss curves
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['train_pixel_acc'], label='train')
    plt.plot(history['val_pixel_acc'], label='val')
    plt.title('Pixel Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_curves.png'))
    plt.close()
    print(f"Saved training curves to '{output_dir}/training_curves.png'")

    # Plot 2: IoU curves
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_iou'], label='Train IoU')
    plt.title('Train IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['val_iou'], label='Val IoU')
    plt.title('Validation IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'iou_curves.png'))
    plt.close()
    print(f"Saved IoU curves to '{output_dir}/iou_curves.png'")

    # Plot 3: Dice curves
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_dice'], label='Train Dice')
    plt.title('Train Dice vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['val_dice'], label='Val Dice')
    plt.title('Validation Dice vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'dice_curves.png'))
    plt.close()
    print(f"Saved Dice curves to '{output_dir}/dice_curves.png'")

    # Plot 4: Combined metrics plot
    plt.figure(figsize=(12, 10))

    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title('Loss vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 2)
    plt.plot(history['train_iou'], label='train')
    plt.plot(history['val_iou'], label='val')
    plt.title('IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 3)
    plt.plot(history['train_dice'], label='train')
    plt.plot(history['val_dice'], label='val')
    plt.title('Dice Score vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 4)
    plt.plot(history['train_pixel_acc'], label='train')
    plt.plot(history['val_pixel_acc'], label='val')
    plt.title('Pixel Accuracy vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Pixel Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'all_metrics_curves.png'))
    plt.close()
    print(f"Saved combined metrics curves to '{output_dir}/all_metrics_curves.png'")


def save_history_to_file(history, output_dir):
    """Save training history to a text file."""
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, 'evaluation_metrics.txt')

    with open(filepath, 'w') as f:
        f.write("TRAINING RESULTS\n")
        f.write("=" * 50 + "\n\n")

        f.write("Final Metrics:\n")
        f.write(f"  Final Train Loss:     {history['train_loss'][-1]:.4f}\n")
        f.write(f"  Final Val Loss:       {history['val_loss'][-1]:.4f}\n")
        f.write(f"  Final Train IoU:      {history['train_iou'][-1]:.4f}\n")
        f.write(f"  Final Val IoU:        {history['val_iou'][-1]:.4f}\n")
        f.write(f"  Final Train Dice:     {history['train_dice'][-1]:.4f}\n")
        f.write(f"  Final Val Dice:       {history['val_dice'][-1]:.4f}\n")
        f.write(f"  Final Train Accuracy: {history['train_pixel_acc'][-1]:.4f}\n")
        f.write(f"  Final Val Accuracy:   {history['val_pixel_acc'][-1]:.4f}\n")
        f.write("=" * 50 + "\n\n")

        f.write("Best Results:\n")
        f.write(f"  Best Val IoU:      {max(history['val_iou']):.4f} (Epoch {np.argmax(history['val_iou']) + 1})\n")
        f.write(f"  Best Val Dice:     {max(history['val_dice']):.4f} (Epoch {np.argmax(history['val_dice']) + 1})\n")
        f.write(f"  Best Val Accuracy: {max(history['val_pixel_acc']):.4f} (Epoch {np.argmax(history['val_pixel_acc']) + 1})\n")
        f.write(f"  Lowest Val Loss:   {min(history['val_loss']):.4f} (Epoch {np.argmin(history['val_loss']) + 1})\n")
        f.write("=" * 50 + "\n\n")

        f.write("Per-Epoch History:\n")
        f.write("-" * 100 + "\n")
        headers = ['Epoch', 'Train Loss', 'Val Loss', 'Train IoU', 'Val IoU',
                   'Train Dice', 'Val Dice', 'Train Acc', 'Val Acc']
        f.write("{:<8} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12}\n".format(*headers))
        f.write("-" * 100 + "\n")

        n_epochs = len(history['train_loss'])
        for i in range(n_epochs):
            f.write("{:<8} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f}\n".format(
                i + 1,
                history['train_loss'][i],
                history['val_loss'][i],
                history['train_iou'][i],
                history['val_iou'][i],
                history['train_dice'][i],
                history['val_dice'][i],
                history['train_pixel_acc'][i],
                history['val_pixel_acc'][i]
            ))

    print(f"Saved evaluation metrics to {filepath}")


# ============================================================================
# Main Training Function
# ============================================================================

def main():
    # Configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Hyperparameters
    batch_size = 50
    w = int(((960 / 2) // 14) * 14)
    h = int(((540 / 2) // 14) * 14)
    lr = 1e-4
    n_epochs = 10

    # Output directory (relative to script location)
    script_dir = os.path.dirname(os.path.abspath(__file__))
    output_dir = os.path.join(script_dir, 'train_stats')
    os.makedirs(output_dir, exist_ok=True)

    # Transforms
    transform = transforms.Compose([
        transforms.Resize((h, w)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    mask_transform = transforms.Compose([
        transforms.Resize((h, w)),
        transforms.ToTensor(),
    ])

    # Dataset paths (relative to script location)
    data_dir = os.path.join(script_dir, '..', 'Offroad_Segmentation_Training_Dataset', 'train')
    val_dir = os.path.join(script_dir, '..', 'Offroad_Segmentation_Training_Dataset', 'val')

    # Create datasets
    trainset = MaskDataset(data_dir=data_dir, transform=transform, mask_transform=mask_transform)
    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)

    valset = MaskDataset(data_dir=val_dir, transform=transform, mask_transform=mask_transform)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)

    print(f"Training samples: {len(trainset)}")
    print(f"Validation samples: {len(valset)}")

    # Load DINOv2 backbone
    print("Loading DINOv2 backbone...")
    BACKBONE_SIZE = "small"
    backbone_archs = {
        "small": "vits14",
        "base": "vitb14_reg",
        "large": "vitl14_reg",
        "giant": "vitg14_reg",
    }
    backbone_arch = backbone_archs[BACKBONE_SIZE]
    backbone_name = f"dinov2_{backbone_arch}"

    backbone_model = torch.hub.load(repo_or_dir="facebookresearch/dinov2", model=backbone_name)
    backbone_model.eval()
    backbone_model.to(device)
    print("Backbone loaded successfully!")

    # Get embedding dimension from backbone
    imgs, _ = next(iter(train_loader))
    imgs = imgs.to(device)
    with torch.no_grad():
        output = backbone_model.forward_features(imgs)["x_norm_patchtokens"]
    n_embedding = output.shape[2]
    print(f"Embedding dimension: {n_embedding}")
    print(f"Patch tokens shape: {output.shape}")

    # Create segmentation head
    classifier = SegmentationHeadConvNeXt(
        in_channels=n_embedding,
        out_channels=n_classes,
        tokenW=w // 14,
        tokenH=h // 14
    )
    classifier = classifier.to(device)

    # Loss and optimizer
    loss_fct = torch.nn.CrossEntropyLoss()
    optimizer = optim.SGD(classifier.parameters(), lr=lr, momentum=0.9)

    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_iou': [],
        'val_iou': [],
        'train_dice': [],
        'val_dice': [],
        'train_pixel_acc': [],
        'val_pixel_acc': []
    }

    # Training loop
    print("\nStarting training...")
    print("=" * 80)

    epoch_pbar = tqdm(range(n_epochs), desc="Training", unit="epoch")
    for epoch in epoch_pbar:
        # Training phase
        classifier.train()
        train_losses = []

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Train]",
                          leave=False, unit="batch")
        for imgs, labels in train_pbar:
            imgs, labels = imgs.to(device), labels.to(device)

            with torch.no_grad():
                output = backbone_model.forward_features(imgs)["x_norm_patchtokens"]

            logits = classifier(output.to(device))
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode="bilinear", align_corners=False)

            labels = labels.squeeze(dim=1).long()

            loss = loss_fct(outputs, labels)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_losses.append(loss.item())
            train_pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Validation phase
        classifier.eval()
        val_losses = []

        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Val]",
                        leave=False, unit="batch")
        with torch.no_grad():
            for imgs, labels in val_pbar:
                imgs, labels = imgs.to(device), labels.to(device)

                output = backbone_model.forward_features(imgs)["x_norm_patchtokens"]
                logits = classifier(output.to(device))
                outputs = F.interpolate(logits, size=imgs.shape[2:], mode="bilinear", align_corners=False)

                labels = labels.squeeze(dim=1).long()
                loss = loss_fct(outputs, labels)
                val_losses.append(loss.item())
                val_pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Calculate metrics
        train_iou, train_dice, train_pixel_acc = evaluate_metrics(
            classifier, backbone_model, train_loader, device, num_classes=n_classes
        )
        val_iou, val_dice, val_pixel_acc = evaluate_metrics(
            classifier, backbone_model, val_loader, device, num_classes=n_classes
        )

        # Store history
        epoch_train_loss = np.mean(train_losses)
        epoch_val_loss = np.mean(val_losses)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_pixel_acc'].append(train_pixel_acc)
        history['val_pixel_acc'].append(val_pixel_acc)

        # Update epoch progress bar with metrics
        epoch_pbar.set_postfix(
            train_loss=f"{epoch_train_loss:.3f}",
            val_loss=f"{epoch_val_loss:.3f}",
            val_iou=f"{val_iou:.3f}",
            val_acc=f"{val_pixel_acc:.3f}"
        )

    # Save plots
    print("\nSaving training curves...")
    save_training_plots(history, output_dir)
    save_history_to_file(history, output_dir)

    # Save model (in scripts directory)
    model_path = os.path.join(script_dir, "segmentation_head.pth")
    torch.save(classifier.state_dict(), model_path)
    print(f"Saved model to '{model_path}'")

    # Final evaluation
    print("\nFinal evaluation results:")
    print(f"  Final Val Loss:     {history['val_loss'][-1]:.4f}")
    print(f"  Final Val IoU:      {history['val_iou'][-1]:.4f}")
    print(f"  Final Val Dice:     {history['val_dice'][-1]:.4f}")
    print(f"  Final Val Accuracy: {history['val_pixel_acc'][-1]:.4f}")

    print("\nTraining complete!")


if __name__ == "__main__":
    main()



Overwriting train_segmentation.py


In [ ]:
%cd /content/Offroad_Segmentation_Scripts
!python train_segmentation.py

/content/Offroad_Segmentation_Scripts
Using device: cuda
Training samples: 2857
Validation samples: 317
Loading DINOv2 backbone...
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Backbone loaded successfully!
Embedding dimension: 384
Patch tokens shape: torch.Size([50, 646, 384])

Starting training...
Training:   0% 0/10 [00:00<?, ?epoch/s]
Epoch 1/10 [Train]:   0% 0/58 [00:00<?, ?batch/s]
Epoch 1/10 [Train]:   0% 0

In [ ]:
!ls /content/Offroad_Segmentation_testImages

Offroad_Segmentation_testImages


In [ ]:
!mv /content/Offroad_Segmentation_testImages/Offroad_Segmentation_testImages/* /content/Offroad_Segmentation_testImages/

In [ ]:
!rm -r /content/Offroad_Segmentation_testImages/Offroad_Segmentation_testImages

In [ ]:
!ls /content/Offroad_Segmentation_testImages

Color_Images  Segmentation


In [ ]:
%cd /content/Offroad_Segmentation_Scripts
!python test_segmentation.py

/content/Offroad_Segmentation_Scripts
Using device: cuda
Loading dataset from /content/Offroad_Segmentation_Scripts/../Offroad_Segmentation_testImages...
Loaded 1002 samples
Loading DINOv2 backbone...
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Backbone loaded successfully!
Embedding dimension: 384
Loading model from /content/Offroad_Segmentation_Scripts/segmentation_head.pth...
Model loaded successfully!

Runni

In [ ]:
!ls

ENV_SETUP    segmentation_head.pth  train_segmentation.py  visualize.py
predictions  test_segmentation.py   train_stats


In [ ]:
from google.colab import files

files.download("train_stats.zip")
files.download("predictions.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r train_stats.zip train_stats
!zip -r predictions.zip predictions

  adding: train_stats/ (stored 0%)
  adding: train_stats/training_curves.png (deflated 7%)
  adding: train_stats/all_metrics_curves.png (deflated 8%)
  adding: train_stats/evaluation_metrics.txt (deflated 74%)
  adding: train_stats/iou_curves.png (deflated 10%)
  adding: train_stats/dice_curves.png (deflated 10%)
  adding: predictions/ (stored 0%)
  adding: predictions/masks/ (stored 0%)
  adding: predictions/masks/0000619_pred.png (deflated 23%)
  adding: predictions/masks/0000872_pred.png (deflated 12%)
  adding: predictions/masks/0000936_pred.png (deflated 4%)
  adding: predictions/masks/0000667_pred.png (deflated 16%)
  adding: predictions/masks/0000832_pred.png (deflated 16%)
  adding: predictions/masks/0000627_pred.png (deflated 6%)
  adding: predictions/masks/0000514_pred.png (deflated 26%)
  adding: predictions/masks/0001030_pred.png (deflated 20%)
  adding: predictions/masks/0000749_pred.png (deflated 23%)
  adding: predictions/masks/0000742_pred.png (deflated 6%)
  adding: pr

In [ ]:
!nano visualize.py

/bin/bash: line 1: nano: command not found


In [ ]:
%%writefile visualize.py
import cv2
import numpy as np
import os
from pathlib import Path

# Input folder containing image files
input_folder = "predictions/masks"
# Output folder for colorized images
output_folder = os.path.join(input_folder, "colorized")

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Get all image files in the folder
image_extensions = ['.png', '.jpg', '.jpeg', '.tiff', '.tif', '.bmp']
image_files = [f for f in Path(input_folder).iterdir()
               if f.is_file() and f.suffix.lower() in image_extensions]

print(f"Found {len(image_files)} image files to process")

# Dictionary to store color mappings (value -> color)
color_map = {}

# Process each file
for image_file in sorted(image_files):
    print(f"Processing: {image_file.name}")

    # Read the image
    im = cv2.imread(str(image_file), cv2.IMREAD_UNCHANGED)

    if im is None:
        print(f"  Skipped: Could not read {image_file.name}")
        continue

    # Get unique values
    u = np.unique(im)

    # Create colorized image
    im2 = np.zeros((im.shape[0], im.shape[1], 3), dtype=np.uint8)

    # Assign colors to each unique value (reuse existing colors if already assigned)
    for v in u:
        if v not in color_map:
            # Generate new random color for this value
            color_map[v] = np.random.randint(0, 255, (3,), dtype=np.uint8)
        im2[im == v] = color_map[v]

    # Save the colorized image
    output_path = os.path.join(output_folder, f"{image_file.stem}.png")
    cv2.imwrite(output_path, im2)
    print(f"  Saved: {output_path}")

print(f"\nProcessing complete! Colorized images saved to: {output_folder}")
print(f"Total unique values found: {len(color_map)}")


Overwriting visualize.py


In [ ]:
!python visualize.py

Found 1002 image files to process
Processing: 0000060_pred.png
  Saved: predictions/masks/colorized/0000060_pred.png
Processing: 0000061_pred.png
  Saved: predictions/masks/colorized/0000061_pred.png
Processing: 0000062_pred.png
  Saved: predictions/masks/colorized/0000062_pred.png
Processing: 0000063_pred.png
  Saved: predictions/masks/colorized/0000063_pred.png
Processing: 0000064_pred.png
  Saved: predictions/masks/colorized/0000064_pred.png
Processing: 0000065_pred.png
  Saved: predictions/masks/colorized/0000065_pred.png
Processing: 0000066_pred.png
  Saved: predictions/masks/colorized/0000066_pred.png
Processing: 0000067_pred.png
  Saved: predictions/masks/colorized/0000067_pred.png
Processing: 0000068_pred.png
  Saved: predictions/masks/colorized/0000068_pred.png
Processing: 0000069_pred.png
  Saved: predictions/masks/colorized/0000069_pred.png
Processing: 0000070_pred.png
  Saved: predictions/masks/colorized/0000070_pred.png
Processing: 0000071_pred.png
  Saved: predictions/mas

In [ ]:
!zip -r colorized_predictions.zip predictions/masks_color

  adding: predictions/masks_color/ (stored 0%)
  adding: predictions/masks_color/0000667_pred_color.png (deflated 39%)
  adding: predictions/masks_color/0000544_pred_color.png (deflated 41%)
  adding: predictions/masks_color/0000110_pred_color.png (deflated 39%)
  adding: predictions/masks_color/0000178_pred_color.png (deflated 40%)
  adding: predictions/masks_color/0000721_pred_color.png (deflated 39%)
  adding: predictions/masks_color/0000863_pred_color.png (deflated 43%)
  adding: predictions/masks_color/0001001_pred_color.png (deflated 44%)
  adding: predictions/masks_color/0000932_pred_color.png (deflated 31%)
  adding: predictions/masks_color/0000712_pred_color.png (deflated 39%)
  adding: predictions/masks_color/0000821_pred_color.png (deflated 34%)
  adding: predictions/masks_color/0000853_pred_color.png (deflated 39%)
  adding: predictions/masks_color/0000297_pred_color.png (deflated 48%)
  adding: predictions/masks_color/0000974_pred_color.png (deflated 33%)
  adding: predict

In [ ]:
%%writefile train_new.py
"""
Segmentation Training Script v3 — Google Colab MAX RESOURCE Edition
=====================================================================
Designed to fully saturate a Colab A100/V100/T4 GPU within a 2-hour window.

Key upgrades vs v2:
  - Effective batch size ~64 via gradient accumulation (physical=8, accum=8)
    → simulates a batch-400 style "large batch" training signal without OOM
  - DINOv2-Large backbone (vitl14_reg, 307M params) — largest practical on Colab
  - Unfreeze last 6 ViT blocks for aggressive domain adaptation
  - Higher input resolution: 518×518 (37×37 token grid, divisible by 14)
  - Wider ASPP (384 channels) + deeper decoder
  - Cosine annealing with warm restarts (SGDR) for better convergence
  - Label smoothing in CrossEntropy
  - Test-Time Augmentation (TTA) at inference: H-flip + scale ensemble
  - Automatic mixed precision + torch.compile (if PyTorch ≥ 2.0)
  - Per-epoch Google Drive checkpoint backup (Colab-safe)
  - Estimated training time: ~90–110 min on A100, ~110–130 min on V100

USAGE (in Colab):
  !python train_segmentation_v3_colab.py \
      --train_dir /content/drive/MyDrive/dataset/train \
      --val_dir   /content/drive/MyDrive/dataset/val \
      --output_dir /content/drive/MyDrive/seg_output \
      --epochs 25
"""

import os, sys, random, argparse, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ============================================================================
# CONFIG  (override via argparse; these are the Colab-optimised defaults)
# ============================================================================
def get_config():
    p = argparse.ArgumentParser(description="Offroad Segmentation v3 — Colab MAX")

    # paths
    p.add_argument("--train_dir",  default="../Offroad_Segmentation_Training_Dataset/train")
    p.add_argument("--val_dir",    default="../Offroad_Segmentation_Training_Dataset/val")
    p.add_argument("--output_dir", default="./seg_output_v3")

    # model
    p.add_argument("--backbone",         default="large",  choices=["small","base","large"])
    p.add_argument("--unfreeze_blocks",  type=int, default=6)
    p.add_argument("--aspp_channels",    type=int, default=384)
    p.add_argument("--img_size",         type=int, default=518)   # must be divisible by 14

    # training
    p.add_argument("--epochs",           type=int,   default=25)
    p.add_argument("--physical_batch",   type=int,   default=8)   # per-GPU batch
    p.add_argument("--accum_steps",      type=int,   default=8)   # effective_batch = physical × accum
    p.add_argument("--lr",               type=float, default=2e-4)
    p.add_argument("--weight_decay",     type=float, default=1e-4)
    p.add_argument("--grad_clip",        type=float, default=1.0)
    p.add_argument("--label_smoothing",  type=float, default=0.1)
    p.add_argument("--dice_weight",      type=float, default=0.5)
    p.add_argument("--num_workers",      type=int,   default=2)
    p.add_argument("--use_compile",      action="store_true", default=False)

    return p.parse_args()


# ============================================================================
# CLASS DEFINITIONS  (11 classes — includes Flowers fix from v2)
# ============================================================================
VALUE_MAP = {
    0: 0, 100: 1, 200: 2, 300: 3, 500: 4,
    550: 5, 600: 6, 700: 7, 800: 8, 7100: 9, 10000: 10,
}
N_CLASSES = 11
CLASS_NAMES = [
    "Background","Trees","Lush Bushes","Dry Grass","Dry Bushes",
    "Ground Clutter","Flowers","Logs","Rocks","Landscape","Sky",
]

# Inverse-frequency weights — rare classes weighted higher
_CW = torch.tensor([0.4,2.0,2.0,1.5,2.0,3.0,4.5,6.0,4.5,0.4,0.4], dtype=torch.float32)
CLASS_WEIGHTS = _CW / _CW.mean()   # keep mean≈1 to stabilise LR

COLOR_PALETTE = np.array([
    [0,0,0],[34,139,34],[0,200,0],[210,180,140],[139,90,43],
    [128,128,0],[255,165,0],[139,69,19],[128,128,128],[160,82,45],[135,206,235],
], dtype=np.uint8)


def convert_mask(mask_img: Image.Image) -> np.ndarray:
    arr = np.array(mask_img, dtype=np.int32)
    out = np.zeros_like(arr, dtype=np.uint8)
    for raw, cls in VALUE_MAP.items():
        out[arr == raw] = cls
    return out   # H×W uint8 class IDs


def mask_to_color(mask_np: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c in range(N_CLASSES):
        rgb[mask_np == c] = COLOR_PALETTE[c]
    return rgb


# ============================================================================
# AUGMENTATION  (joint spatial + photometric)
# ============================================================================
class JointAugment:
    def __init__(self, size: int, is_train: bool):
        self.size     = size
        self.is_train = is_train

    def __call__(self, img: Image.Image, mask_np: np.ndarray):
        mask_img = Image.fromarray(mask_np)   # PIL for spatial ops

        # ── resize ──
        img      = TF.resize(img,      (self.size, self.size), interpolation=Image.BILINEAR)
        mask_img = TF.resize(mask_img, (self.size, self.size), interpolation=Image.NEAREST)

        if self.is_train:
            # Random horizontal flip
            if random.random() > 0.5:
                img = TF.hflip(img); mask_img = TF.hflip(mask_img)

            # Random vertical flip (synthetic data → valid)
            if random.random() > 0.8:
                img = TF.vflip(img); mask_img = TF.vflip(mask_img)

            # Multi-scale random crop (75%–100%)
            scale    = random.uniform(0.75, 1.0)
            ch, cw   = int(self.size * scale), int(self.size * scale)
            i, j, h, w = transforms.RandomCrop.get_params(img, (ch, cw))
            img      = TF.resized_crop(img,      i, j, h, w, (self.size, self.size), Image.BILINEAR)
            mask_img = TF.resized_crop(mask_img, i, j, h, w, (self.size, self.size), Image.NEAREST)

            # Random 90° rotation (0, 90, 180, 270)
            angle = random.choice([0, 90, 180, 270])
            if angle:
                img      = TF.rotate(img,      angle)
                mask_img = TF.rotate(mask_img, angle)

            # Photometric augmentation (image only)
            img = TF.adjust_brightness(img, random.uniform(0.6, 1.4))
            img = TF.adjust_contrast(img,   random.uniform(0.6, 1.4))
            img = TF.adjust_saturation(img, random.uniform(0.6, 1.4))
            img = TF.adjust_hue(img,        random.uniform(-0.15, 0.15))
            if random.random() > 0.5:
                img = TF.gaussian_blur(img, kernel_size=5,
                                       sigma=random.uniform(0.1, 2.0))
            # Random grayscale (forces texture reliance)
            if random.random() > 0.8:
                img = TF.to_grayscale(img, num_output_channels=3)

        # ── to tensor ──
        img_t  = TF.to_tensor(img)
        img_t  = TF.normalize(img_t, [0.485,0.456,0.406], [0.229,0.224,0.225])
        mask_t = torch.from_numpy(np.array(mask_img)).long()
        return img_t, mask_t


# ============================================================================
# DATASET
# ============================================================================
class SegDataset(Dataset):
    def __init__(self, data_dir: str, augment: JointAugment):
        self.img_dir  = os.path.join(data_dir, "Color_Images")
        self.msk_dir  = os.path.join(data_dir, "Segmentation")
        self.augment  = augment
        self.ids      = sorted(f for f in os.listdir(self.img_dir)
                               if f.lower().endswith((".png",".jpg",".jpeg")))
        assert len(self.ids) > 0, f"No images found in {self.img_dir}"

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]
        img  = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        msk  = Image.open(os.path.join(self.msk_dir, name))
        msk_np = convert_mask(msk)
        img_t, msk_t = self.augment(img, msk_np)
        return img_t, msk_t


# ============================================================================
# MODEL — DINOv2 + DeepLabV3+ head (WIDER)
# ============================================================================
class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling with configurable channel width."""
    def __init__(self, in_ch: int, out_ch: int = 384,
                 dilations: tuple = (1, 6, 12, 18, 24)):
        super().__init__()
        self.branches = nn.ModuleList()
        for d in dilations:
            ks = 1 if d == 1 else 3
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, ks, padding=d if d > 1 else 0,
                          dilation=d, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ))
        # GAP branch
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        total_in = out_ch * (len(dilations) + 1)
        self.project = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout2d(0.3),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [b(x) for b in self.branches]
        feats.append(F.interpolate(self.gap(x), (h,w),
                                   mode="bilinear", align_corners=False))
        return self.project(torch.cat(feats, dim=1))


class DeepLabV3PlusHead(nn.Module):
    """
    Upgraded DeepLabV3+ head:
      - Wider ASPP (5 dilation rates instead of 4)
      - 3-layer decoder with residual connection
      - Squeeze-and-Excitation on ASPP output
    """
    def __init__(self, in_ch: int, num_classes: int,
                 token_h: int, token_w: int,
                 aspp_ch: int = 384):
        super().__init__()
        self.token_h = token_h
        self.token_w = token_w

        self.input_proj = nn.Sequential(
            nn.Conv2d(in_ch, aspp_ch, 1, bias=False),
            nn.BatchNorm2d(aspp_ch), nn.ReLU(inplace=True),
        )
        self.aspp = ASPP(aspp_ch, aspp_ch)

        # Squeeze-and-Excitation on ASPP output (channel attention)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(aspp_ch, aspp_ch // 16),
            nn.ReLU(inplace=True),
            nn.Linear(aspp_ch // 16, aspp_ch),
            nn.Sigmoid(),
        )

        # 3-conv decoder
        self.dec1 = nn.Sequential(
            nn.Conv2d(aspp_ch, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.dec3 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(128, num_classes, 1)

        # shortcut for residual
        self.shortcut = nn.Conv2d(aspp_ch, 256, 1, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.token_h, self.token_w, C).permute(0, 3, 1, 2)
        x = self.input_proj(x)
        x = self.aspp(x)

        # SE channel attention
        se = self.se(x).view(x.size(0), x.size(1), 1, 1)
        x  = x * se

        res = self.shortcut(x)
        x   = self.dec1(x)
        x   = self.dec2(x) + res   # residual
        x   = self.dec3(x)
        return self.classifier(x)


# ============================================================================
# LOSS  (CE with label smoothing + Dice)
# ============================================================================
class CombinedLoss(nn.Module):
    def __init__(self, class_weights=None, dice_weight=0.5,
                 label_smoothing=0.1, smooth=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=label_smoothing,
            ignore_index=255,
        )
        self.dice_w = dice_weight
        self.smooth = smooth

    def _dice(self, logits, targets):
        num_cls = logits.shape[1]
        probs   = F.softmax(logits, dim=1)
        t_oh    = F.one_hot(targets.clamp(0, num_cls-1), num_cls) \
                    .permute(0,3,1,2).float()
        dims    = (0, 2, 3)
        inter   = (probs * t_oh).sum(dims)
        card    = (probs + t_oh).sum(dims)
        return (1.0 - (2*inter + self.smooth) / (card + self.smooth)).mean()

    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice_w * self._dice(logits, targets)


# ============================================================================
# METRICS
# ============================================================================
def iou_per_class(pred_logits, target):
    """Returns list of per-class IoU (nan if class absent in batch)."""
    pred = torch.argmax(pred_logits, dim=1).view(-1)
    tgt  = target.view(-1)
    ious = []
    for c in range(N_CLASSES):
        p = pred == c; t = tgt == c
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        ious.append((inter/union).item() if union > 0 else float("nan"))
    return ious


def mean_iou(ious):
    v = [x for x in ious if not math.isnan(x)]
    return float(np.mean(v)) if v else 0.0


def pixel_acc(pred_logits, target):
    return (torch.argmax(pred_logits,1) == target).float().mean().item()


# ============================================================================
# TEST-TIME AUGMENTATION (TTA)
# ============================================================================
@torch.no_grad()
def tta_predict(backbone, head, imgs, size):
    """Average predictions over original + H-flip."""
    feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
    logits = head(feats)
    logits = F.interpolate(logits, size, mode="bilinear", align_corners=False)

    imgs_f  = torch.flip(imgs, dims=[3])
    feats_f = backbone.forward_features(imgs_f)["x_norm_patchtokens"]
    logits_f = head(feats_f)
    logits_f = F.interpolate(logits_f, size, mode="bilinear", align_corners=False)
    logits_f = torch.flip(logits_f, dims=[3])

    return (F.softmax(logits,1) + F.softmax(logits_f,1)) / 2.0


# ============================================================================
# EVALUATION
# ============================================================================
@torch.no_grad()
def evaluate(backbone, head, loader, device, loss_fn, use_amp, use_tta=False):
    backbone.eval(); head.eval()
    size = (loader.dataset.augment.size,) * 2
    tot_loss = tot_acc = 0.0
    all_ious = [[] for _ in range(N_CLASSES)]

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            if use_tta:
                probs  = tta_predict(backbone, head, imgs, size)
                logits_for_loss = torch.log(probs.clamp(min=1e-7))
                loss   = F.nll_loss(logits_for_loss, labels,
                                    weight=loss_fn.ce.weight)
                out    = probs
            else:
                feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
                logits = head(feats)
                logits = F.interpolate(logits, size,
                                       mode="bilinear", align_corners=False)
                loss   = loss_fn(logits, labels)
                out    = logits

        tot_loss += loss.item()
        tot_acc  += pixel_acc(out, labels)
        for c, v in enumerate(iou_per_class(out, labels)):
            all_ious[c].append(v)

    n = max(len(loader), 1)
    class_ious = [mean_iou(all_ious[c]) for c in range(N_CLASSES)]
    backbone.train(); head.train()
    return tot_loss/n, tot_acc/n, class_ious


# ============================================================================
# PLOTTING
# ============================================================================
def save_plots(history, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ep = range(1, len(history["train_loss"])+1)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    axes[0,0].plot(ep, history["train_loss"], label="train")
    axes[0,0].plot(ep, history["val_loss"],   label="val")
    axes[0,0].set_title("Loss (CE + Dice)"); axes[0,0].legend(); axes[0,0].grid()

    axes[0,1].plot(ep, history["train_iou"], label="train")
    axes[0,1].plot(ep, history["val_iou"],   label="val")
    axes[0,1].set_title("Mean IoU"); axes[0,1].legend(); axes[0,1].grid()
    axes[0,1].axhline(0.6, color="red", linestyle="--", alpha=0.5, label="target 0.60")

    axes[1,0].plot(ep, history["train_acc"], label="train")
    axes[1,0].plot(ep, history["val_acc"],   label="val")
    axes[1,0].set_title("Pixel Accuracy"); axes[1,0].legend(); axes[1,0].grid()

    final_iou = history["val_class_ious"][-1]
    axes[1,1].barh(range(N_CLASSES), final_iou,
                   color=[COLOR_PALETTE[c]/255 for c in range(N_CLASSES)])
    axes[1,1].set_yticks(range(N_CLASSES))
    axes[1,1].set_yticklabels(CLASS_NAMES, fontsize=8)
    axes[1,1].set_xlim(0,1)
    axes[1,1].axvline(mean_iou(final_iou), color="red",
                      linestyle="--", label=f"mean={mean_iou(final_iou):.3f}")
    axes[1,1].set_title("Per-Class Val IoU (final epoch)")
    axes[1,1].legend(); axes[1,1].grid(axis="x")

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "training_dashboard.png"), dpi=150)
    plt.close()


def save_metrics_txt(history, cfg, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    best_ep = int(np.argmax(history["val_iou"]))
    eff_bs  = cfg.physical_batch * cfg.accum_steps
    with open(os.path.join(out_dir, "evaluation_metrics.txt"), "w") as f:
        f.write("TRAINING RESULTS — v3 Colab MAX\n")
        f.write("=" * 65 + "\n")
        f.write(f"Backbone:           DINOv2-{cfg.backbone.capitalize()}\n")
        f.write(f"Effective batch:    {eff_bs}  "
                f"(physical={cfg.physical_batch} × accum={cfg.accum_steps})\n")
        f.write(f"Input resolution:   {cfg.img_size}×{cfg.img_size}\n")
        f.write(f"Unfrozen blocks:    {cfg.unfreeze_blocks}\n")
        f.write(f"Epochs trained:     {len(history['train_loss'])}\n")
        f.write("=" * 65 + "\n\n")
        f.write(f"Best Val IoU:       {max(history['val_iou']):.4f}  (epoch {best_ep+1})\n")
        f.write(f"Final Val IoU:      {history['val_iou'][-1]:.4f}\n")
        f.write(f"Final Val Acc:      {history['val_acc'][-1]:.4f}\n")
        f.write(f"Final Val Loss:     {history['val_loss'][-1]:.4f}\n\n")
        f.write("Per-Class Val IoU @ best epoch:\n")
        for name, iou in zip(CLASS_NAMES, history["val_class_ious"][best_ep]):
            bar = "█" * int(iou * 30)
            f.write(f"  {name:<20} {iou:.4f}  {bar}\n")
        f.write("\n" + "=" * 65 + "\n")
        f.write("Per-Epoch History:\n")
        f.write(f"{'Ep':>4} {'TrLoss':>9} {'VaLoss':>9} "
                f"{'TrIoU':>8} {'VaIoU':>8} {'VaAcc':>8}\n")
        f.write("-" * 50 + "\n")
        for i in range(len(history["train_loss"])):
            f.write(f"{i+1:>4} {history['train_loss'][i]:>9.4f} "
                    f"{history['val_loss'][i]:>9.4f} "
                    f"{history['train_iou'][i]:>8.4f} "
                    f"{history['val_iou'][i]:>8.4f} "
                    f"{history['val_acc'][i]:>8.4f}\n")
    print(f"✓ Metrics saved to {out_dir}/evaluation_metrics.txt")


# ============================================================================
# MAIN
# ============================================================================
def main():
    cfg    = get_config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    os.makedirs(cfg.output_dir, exist_ok=True)

    print("=" * 65)
    print("Offroad Segmentation v3 — Colab MAX")
    print("=" * 65)
    print(f"  Device:            {device}")
    if device.type == "cuda":
        prop = torch.cuda.get_device_properties(0)
        print(f"  GPU:               {prop.name}  ({prop.total_memory//1024**3} GB)")
    print(f"  Backbone:          DINOv2-{cfg.backbone.capitalize()}")
    print(f"  Effective batch:   {cfg.physical_batch * cfg.accum_steps}")
    print(f"  Input resolution:  {cfg.img_size}×{cfg.img_size}")
    print(f"  Epochs:            {cfg.epochs}")
    print(f"  AMP:               {use_amp}")
    print("=" * 65)

    S = cfg.img_size
    assert S % 14 == 0, f"img_size ({S}) must be divisible by 14"

    # ── datasets ──────────────────────────────────────────────────────────────
    train_aug = JointAugment(S, is_train=True)
    val_aug   = JointAugment(S, is_train=False)

    train_set = SegDataset(cfg.train_dir, train_aug)
    val_set   = SegDataset(cfg.val_dir,   val_aug)

    train_loader = DataLoader(
        train_set, batch_size=cfg.physical_batch,
        shuffle=True,  num_workers=cfg.num_workers,
        pin_memory=True, drop_last=True,
        persistent_workers=(cfg.num_workers > 0),
    )
    val_loader = DataLoader(
        val_set, batch_size=cfg.physical_batch,
        shuffle=False, num_workers=cfg.num_workers,
        pin_memory=True,
        persistent_workers=(cfg.num_workers > 0),
    )
    print(f"\nTrain: {len(train_set)} images  |  Val: {len(val_set)} images")

    # ── backbone ──────────────────────────────────────────────────────────────
    backbone_names = {
        "small": "dinov2_vits14",
        "base":  "dinov2_vitb14_reg",
        "large": "dinov2_vitl14_reg",
    }
    print(f"\nLoading {backbone_names[cfg.backbone]} ...")
    backbone = torch.hub.load("facebookresearch/dinov2",
                              backbone_names[cfg.backbone],
                              pretrained=True)
    backbone.to(device)

    # Freeze all, then unfreeze last N blocks + norm
    for p in backbone.parameters(): p.requires_grad_(False)
    for block in list(backbone.blocks)[-cfg.unfreeze_blocks:]:
        for p in block.parameters(): p.requires_grad_(True)
    for p in backbone.norm.parameters(): p.requires_grad_(True)
    n_trainable_bb = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print(f"Backbone trainable params: {n_trainable_bb:,}")

    # Probe feature dimensions
    with torch.no_grad():
        probe = torch.zeros(1, 3, S, S, device=device)
        feats = backbone.forward_features(probe)["x_norm_patchtokens"]
    embed_dim         = feats.shape[-1]
    token_h = token_w = S // 14
    print(f"Embed dim: {embed_dim}  |  Token grid: {token_h}×{token_w}")

    # ── segmentation head ──────────────────────────────────────────────────
    head = DeepLabV3PlusHead(
        embed_dim, N_CLASSES, token_h, token_w,
        aspp_ch=cfg.aspp_channels,
    ).to(device)
    n_head = sum(p.numel() for p in head.parameters())
    print(f"Head params: {n_head:,}")

    # Optional torch.compile (PyTorch ≥ 2.0, Colab Pro usually has it)
    if cfg.use_compile and hasattr(torch, "compile"):
        print("Compiling head with torch.compile ...")
        head = torch.compile(head)

    # ── loss ──────────────────────────────────────────────────────────────────
    loss_fn = CombinedLoss(
        class_weights=CLASS_WEIGHTS.to(device),
        dice_weight=cfg.dice_weight,
        label_smoothing=cfg.label_smoothing,
    )

    # ── optimizer & scheduler ─────────────────────────────────────────────────
    # Differential LR: backbone gets 10× less than head
    param_groups = [
        {"params": [p for p in backbone.parameters() if p.requires_grad],
         "lr": cfg.lr * 0.05,    # 5% LR for backbone — conservative fine-tune
         "name": "backbone"},
        {"params": list(head.parameters()),
         "lr": cfg.lr,
         "name": "head"},
    ]
    optimizer = optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    # Cosine annealing with warm restarts — 2 restarts over the run
    T0     = max(cfg.epochs // 3, 5)
    sched  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=2, eta_min=1e-6,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    eff_bs = cfg.physical_batch * cfg.accum_steps
    print(f"\nEffective batch size: {eff_bs}  "
          f"(physical={cfg.physical_batch} × accum={cfg.accum_steps})")
    print(f"Steps per epoch: {len(train_loader)}  "
          f"(≈{len(train_loader)//cfg.accum_steps} optimizer steps)")
    print(f"\nStarting training for {cfg.epochs} epochs ...\n")

    best_val_iou  = 0.0
    ckpt_path     = os.path.join(cfg.output_dir, "best_model_v3.pth")
    history       = {k: [] for k in [
        "train_loss","val_loss","train_iou","val_iou",
        "train_acc","val_acc","val_class_ious",
    ]}

    for epoch in range(cfg.epochs):
        t0 = time.time()
        backbone.train(); head.train()

        ep_loss, ep_iou, ep_acc = [], [], []
        optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader),
                    total=len(train_loader),
                    desc=f"Epoch {epoch+1:>3}/{cfg.epochs} [Train]",
                    leave=False, ncols=100)

        for step, (imgs, labels) in pbar:
            imgs, labels = imgs.to(device, non_blocking=True), \
                           labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                feats  = backbone.forward_features(imgs)["x_norm_patchtokens"]
                logits = head(feats)
                logits = F.interpolate(logits, (S, S),
                                       mode="bilinear", align_corners=False)
                # Scale loss by accum_steps so effective gradient magnitude
                # matches a true large-batch run
                loss   = loss_fn(logits, labels) / cfg.accum_steps

            scaler.scale(loss).backward()

            # Gradient accumulation step
            if (step + 1) % cfg.accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for pg in optimizer.param_groups for p in pg["params"]],
                    cfg.grad_clip,
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            with torch.no_grad():
                real_loss = loss.item() * cfg.accum_steps
                batch_iou = mean_iou(iou_per_class(logits, labels))
                batch_acc = pixel_acc(logits, labels)

            ep_loss.append(real_loss)
            ep_iou.append(batch_iou)
            ep_acc.append(batch_acc)
            pbar.set_postfix(
                loss=f"{real_loss:.4f}",
                iou=f"{batch_iou:.3f}",
                lr=f"{optimizer.param_groups[1]['lr']:.2e}",
            )

        sched.step(epoch)   # per-epoch step for CosineAnnealingWarmRestarts

        # ── validation ──
        val_loss, val_acc, val_ciu = evaluate(
            backbone, head, val_loader, device, loss_fn, use_amp,
            use_tta=(epoch >= cfg.epochs - 5),   # TTA only in last 5 epochs
        )
        val_iou = mean_iou(val_ciu)
        elapsed = time.time() - t0

        history["train_loss"].append(float(np.mean(ep_loss)))
        history["val_loss"].append(val_loss)
        history["train_iou"].append(float(np.mean(ep_iou)))
        history["val_iou"].append(val_iou)
        history["train_acc"].append(float(np.mean(ep_acc)))
        history["val_acc"].append(val_acc)
        history["val_class_ious"].append(val_ciu)

        print(f"Epoch {epoch+1:>3}/{cfg.epochs}  "
              f"tr_loss={history['train_loss'][-1]:.4f}  "
              f"va_loss={val_loss:.4f}  "
              f"tr_iou={history['train_iou'][-1]:.4f}  "
              f"va_iou={val_iou:.4f}  "
              f"va_acc={val_acc:.4f}  "
              f"[{elapsed:.0f}s]")

        # Per-class breakdown every 5 epochs
        if (epoch + 1) % 5 == 0:
            print("  ┌─ Per-class Val IoU:")
            for name, iou in zip(CLASS_NAMES, val_ciu):
                bar = "█" * int(iou * 25)
                print(f"  │ {name:<20} {iou:.3f}  {bar}")
            print("  └─")

        # Save best checkpoint
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save({
                "epoch":             epoch + 1,
                "head_state_dict":   head.state_dict()
                                     if not hasattr(head, "_orig_mod")
                                     else head._orig_mod.state_dict(),
                "val_iou":           val_iou,
                "val_class_ious":    val_ciu,
                "config": {
                    "backbone":       cfg.backbone,
                    "img_size":       cfg.img_size,
                    "aspp_channels":  cfg.aspp_channels,
                    "embed_dim":      embed_dim,
                    "token_h":        token_h,
                    "token_w":        token_w,
                    "n_classes":      N_CLASSES,
                },
            }, ckpt_path)
            print(f"  ✓ Best checkpoint saved  (val_iou={val_iou:.4f})")

        # Intermediate save every 5 epochs (Colab safety net)
        if (epoch + 1) % 5 == 0:
            inter_path = os.path.join(cfg.output_dir,
                                      f"checkpoint_ep{epoch+1}.pth")
            torch.save({"epoch": epoch+1,
                        "head_state_dict": head.state_dict()
                                           if not hasattr(head, "_orig_mod")
                                           else head._orig_mod.state_dict(),
                        "val_iou": val_iou}, inter_path)
            print(f"  ✓ Intermediate checkpoint: {inter_path}")

    # ── final saves ──────────────────────────────────────────────────────────
    save_plots(history, cfg.output_dir)
    save_metrics_txt(history, cfg, cfg.output_dir)

    print("\n" + "=" * 65)
    print("Training complete!")
    print(f"Best Val IoU:   {best_val_iou:.4f}")
    print(f"Checkpoint:     {ckpt_path}")
    print(f"Output dir:     {cfg.output_dir}")
    print("=" * 65)


if __name__ == "__main__":
    main()


Overwriting train_new.py


In [ ]:
!ls

' '			     predictions.zip	     train_segmentation.py
 colorized_predictions.zip   segmentation_head.pth   train_stats
 ENV_SETUP		     test_segmentation.py    train_stats.zip
 predictions		     train_new.py	     visualize.py


In [ ]:
!python  train_new.py

Offroad Segmentation v3 — Colab MAX
  Device:            cuda
  GPU:               Tesla T4  (14 GB)
  Backbone:          DINOv2-Large
  Effective batch:   64
  Input resolution:  518×518
  Epochs:            25
  AMP:               True

Train: 2857 images  |  Val: 317 images

Loading dinov2_vitl14_reg ...
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Backbone trainable params: 75,591,680
Embed dim: 1024  |  Toke